In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [ ]:
df = pd.read_csv("Country-data.csv")

df.head()

In [ ]:
print(df.shape)

df.info()

print(df.isnull().sum())

print(df.duplicated().sum())

df.describe()

In [ ]:
df.hist(figsize=(15,10))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15,10))

for i, col in enumerate(df.columns[1:]):
    plt.subplot(3,3,i+1)
    sns.boxplot(y=df[col])

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,8))

sns.heatmap(
    df.drop('country', axis=1).corr(),
    annot=True,
    cmap='coolwarm'
)

plt.title("Correlation Heatmap")
plt.show()

In [ ]:
X = df.drop('country', axis=1)

In [ ]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

In [ ]:
pca = PCA(n_components=0.90)

X_pca = pca.fit_transform(X_scaled)

print(X_pca.shape)

print(pca.explained_variance_ratio_)

print(np.sum(pca.explained_variance_ratio_))

In [ ]:
wcss = []

for i in range(1,11):

    kmeans = KMeans(
        n_clusters=i,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X_pca)

    wcss.append(kmeans.inertia_)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(range(1,11), wcss, marker='o')

plt.xlabel("Number of Clusters")
plt.ylabel("WCSS")
plt.title("Elbow Method")

plt.show()

In [ ]:
for i in range(2,11):

    kmeans = KMeans(
        n_clusters=i,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_pca)

    score = silhouette_score(X_pca, labels)

    print(f"Clusters: {i}, Silhouette Score: {score}")

In [ ]:
kmeans = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

clusters = kmeans.fit_predict(X_pca)

df['cluster'] = clusters

print(df['cluster'].value_counts())

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=clusters,
    cmap='viridis'
)

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title("Country Clusters")

plt.show()

In [ ]:
cluster_summary = df.groupby('cluster').mean(numeric_only=True)

cluster_summary

In [ ]:
poor_cluster = cluster_summary['gdpp'].idxmin()

print(poor_cluster)

In [ ]:
target_countries = df[df['cluster'] == poor_cluster]

target_countries = target_countries.sort_values(
    by=['gdpp', 'income']
)

target_countries[
    ['country', 'gdpp', 'income', 'child_mort', 'life_expec']
].head(20)

In [ ]:
target_countries.to_csv(
    "countries_needing_aid.csv",
    index=False
)

print("Results saved successfully.")